# Predicting gene knockout strategies

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you should be able to:

1. Explain what *growth-coupled* production means and recognise it in a production envelope.
2. Evaluate a hand-made knockout design by comparing the production envelopes of wild type and mutant.
3. Translate reaction knockouts into gene knockouts using gene-protein-reaction (GPR) rules, and detect side effects.
4. Run OptGene and OptKnock in cameo and interpret the knockout sets they return.

**Estimated time:** 90 min (including exercises).

**Prerequisites:** FBA and the phenotypic phase plane / production envelope (notebooks 04 and 07), knocking out reactions and genes (notebooks 05 and 06).

### Background

A cell does not care about our product. FBA assumes it maximises growth, so any carbon that goes into a product is carbon "lost" from biomass. If the wild type can grow at its maximum rate without making any succinate, evolution and the FBA optimum both push production towards zero.

**Growth coupling** changes this. We remove reactions so that the only way the network can balance its cofactors (NADH, ATP) at high growth is by secreting the product. In the production envelope, a growth-coupled design shows a *minimum* product flux above zero at the maximum growth rate: the cell makes product because it has to, not because we ask it to. Growth-coupled strains are attractive because adaptive laboratory evolution for faster growth then also improves production.

Finding good knockout sets by hand is hard, because the number of combinations explodes (choosing 3 out of ~95 reactions in the core model already gives more than 100 000 sets). Two classic algorithms automate the search:

* **OptGene** ([Patil et al., 2005](https://doi.org/10.1186/1471-2105-6-308)) uses an *evolutionary algorithm*: a population of candidate knockout sets is mutated and recombined, and each candidate is scored by simulating the mutant (FBA or MOMA) and computing a fitness such as the biomass-product coupled yield. It is a heuristic: fast and flexible, but it gives no guarantee of optimality and results vary between runs.
* **OptKnock** ([Burgard et al., 2003](https://doi.org/10.1002/bit.10803)) formulates the problem as a *bilevel optimisation*: the outer problem (the engineer) chooses knockouts to maximise product flux, while the inner problem (the cell) maximises growth. Using LP duality the inner problem is folded into a single mixed-integer linear program (MILP), which a solver can solve to optimality.

MILPs are much harder than LPs. The original version of this course used the commercial solver CPLEX; here we use the free GLPK solver, which is considerably slower on MILPs (cameo even prints a warning). We therefore keep the number of knockouts small and use the small *E. coli* core model.

Both methods are implemented in cameo ([Cardoso et al., 2018](https://doi.org/10.1021/acssynbio.7b00423)).

In [ ]:
from cobra.io import read_sbml_model
import cameo_compat  # noqa: F401 (patches cameo for modern Python/cobra)
from cameo import phenotypic_phase_plane
from cameo.visualization.plotting.with_plotly import PlotlyPlotter
plotter = PlotlyPlotter()

In [ ]:
model = read_sbml_model('data/e_coli_core.xml.gz')

We simulate **anaerobic** growth by closing the oxygen exchange (lower bound 0 means no uptake). Under anaerobic conditions *E. coli* has to regenerate NAD⁺ by fermentation, which makes succinate (a reduced product) a natural candidate for coupling.

In [ ]:
model.reactions.EX_o2_e.lower_bound = 0

`EX_succ_e` is the exchange reaction of extracellular succinate. A positive flux through it means succinate is secreted.

In [ ]:
model.reactions.EX_succ_e

### Wild-type production envelope

The phenotypic phase plane (production envelope) shows, for every feasible growth rate on the x-axis, the minimum and maximum succinate secretion the network allows. Look at the right-most point (maximum growth): if the lower curve touches zero there, the wild type can grow optimally without producing any succinate, so production is **not** growth coupled.

In [ ]:
ppp = phenotypic_phase_plane(model,
                             variables=[model.reactions.BIOMASS_Ecoli_core_w_GAM],
                             objective=model.reactions.EX_succ_e)

ppp.plot(plotter)

### Design 1: knock out ACKr, ATPS4r, ATPM and FUM

`model.copy()` creates an independent mutant, so the wild-type model stays unchanged. We knock out acetate kinase (ACKr), ATP synthase (ATPS4r), the non-growth ATP maintenance reaction (ATPM) and fumarase (FUM). Note that knocking out ATPM is not biologically meaningful: it is a modelling pseudo-reaction for the maintenance energy of the cell, not an enzyme.

In [ ]:
mutant1 = model.copy()

In [ ]:
mutant1.reactions.ACKr.knock_out()
mutant1.reactions.ATPS4r.knock_out()
mutant1.reactions.ATPM.knock_out()
mutant1.reactions.FUM.knock_out()

In [ ]:
ppp = phenotypic_phase_plane(mutant1,
                             variables=[mutant1.reactions.BIOMASS_Ecoli_core_w_GAM],
                             objective=mutant1.reactions.EX_succ_e)

ppp.plot(plotter)

### Design 2: knock out ACALD, PYK and ME2

Acetaldehyde dehydrogenase (ACALD, part of the ethanol pathway), pyruvate kinase (PYK) and NADP-dependent malic enzyme (ME2). Removing competing fermentation routes forces carbon towards the reductive branch of the TCA cycle, which ends in succinate.

In [ ]:
mutant2 = model.copy()

In [ ]:
mutant2.reactions.ACALD.knock_out()
mutant2.reactions.PYK.knock_out()
mutant2.reactions.ME2.knock_out()

In [ ]:
ppp = phenotypic_phase_plane(mutant2,
                             variables=[mutant2.reactions.BIOMASS_Ecoli_core_w_GAM],
                             objective=mutant2.reactions.EX_succ_e)

ppp.plot(plotter)

### Design 3: knock out ACALD and LDH_D

A smaller design: block ethanol (ACALD) and D-lactate (LDH_D) formation, the two main NADH-consuming fermentation routes. Compare the minimum succinate flux at maximum growth with design 2.

In [ ]:
mutant3 = model.copy()

In [ ]:
mutant3.reactions.ACALD.knock_out()
mutant3.reactions.LDH_D.knock_out()

In [ ]:
ppp = phenotypic_phase_plane(mutant3,
                             variables=[mutant3.reactions.BIOMASS_Ecoli_core_w_GAM],
                             objective=mutant3.reactions.EX_succ_e)

ppp.plot(plotter)

We can also knock-out genes instead, and implicitly block the reactions that are annotated to only occur in presence of the corresponding gene. To reproduce the ACALD, LDH_D knock-out above with genes instead we can do:

In [ ]:
mutant4 = model.copy()

In [ ]:
to_knock = mutant4.reactions.ACALD.genes.union(mutant4.reactions.LDH_D.genes)
for gene in to_knock:
    gene.knock_out()

In [ ]:
ppp = phenotypic_phase_plane(mutant4,
                             variables=[mutant4.reactions.BIOMASS_Ecoli_core_w_GAM],
                             objective=mutant4.reactions.EX_succ_e)

ppp.plot(plotter)

The envelope is identical to design 3: in the core model the four genes remove exactly the two intended reactions. Exercise 2 checks whether that is always true.

## Exercises (20 min)

1. Do any of these designs also work (are they growth coupled) under **aerobic** conditions? Hint: re-open the oxygen exchange (`lower_bound = -1000`) on a copy of each design and look at the minimum succinate flux at the maximum growth rate.
2. Did the gene-knockout design (`mutant4`) have any unintended effects? Hint: after knocking out the genes, list all reactions for which `reaction.functional` is `False`. Repeat the check in the genome-scale model `data/iJO1366.xml.gz`.

In [ ]:
# Your code here


# Predicting gene knockout strategies with algorithms

Instead of guessing knockouts by hand, cameo can search for them: with an evolutionary algorithm (**OptGene**) or with mixed-integer linear programming (**OptKnock**).

## OptGene

OptGene searches for gene or reaction knockouts with an evolutionary algorithm ([Patil et al., 2005](https://doi.org/10.1186/1471-2105-6-308)). Each individual in the population is a set of knockouts. Its fitness is computed by simulating the mutant (here with FBA) and scoring the result with the biomass-product coupled yield (BPCY: growth rate times product yield per substrate). With `growth_coupled=True`, as below, cameo uses the *minimum* product yield from flux variability analysis instead, so only designs that force production score well. The best individuals are mutated and recombined into the next generation, until `max_evaluations` simulations have been done. The figure from the original paper summarises the workflow:

<img src="https://static-content.springer.com/image/art%3A10.1186%2F1471-2105-6-308/MediaObjects/12859_2005_Article_632_Fig1_HTML.jpg" alt="OptGene workflow: an initial population of knockout sets is evaluated by metabolic simulation, scored by a fitness function, and evolved by selection, crossover and mutation" width="600"/>

We reload a fresh (aerobic) core model and ask for up to five knockouts that couple succinate secretion to growth on glucose. The search is stochastic; `seed=42` makes it reproducible. Change the seed to see how the proposed designs vary.

In [ ]:
from cameo.strain_design.heuristic.evolutionary_based import OptGene

In [ ]:
from cobra.io import read_sbml_model
model = read_sbml_model('data/e_coli_core.xml.gz')

In [ ]:
optgene = OptGene(model)

In [ ]:
result = optgene.run(target=model.reactions.EX_succ_e, 
                     biomass=model.reactions.BIOMASS_Ecoli_core_w_GAM,
                     substrate=model.metabolites.glc__D_e,
                     max_evaluations=20000, population_size=200, max_knockouts=5,
                     plot=False, growth_coupled=True, seed=42)

In [ ]:
result

The result table lists each knockout set, the corresponding genes, the simulated growth rate and product flux, and the fitness. Several different knockout sets can reach similar fitness.

<div class="alert alert-warning">

Without restrictions, OptGene may propose knockouts that are not realistic genetic targets, for example ATP synthase (ATPS4r) or `s0001`, a placeholder "gene" the model uses for spontaneous (non-enzymatic) reactions. Which designs appear depends on the random seed.

</div>

<div class="alert alert-success">
Workaround: Use the `essential_genes` parameter to `OptGene` to create an exclude list.
</div>

In [ ]:
essential = [g.id for g in model.reactions.ATPS4r.genes] + ['s0001']
optgene = OptGene(model, essential_genes=essential)
result = optgene.run(target=model.reactions.EX_succ_e, 
                     biomass=model.reactions.BIOMASS_Ecoli_core_w_GAM,
                     substrate=model.metabolites.glc__D_e,
                     max_evaluations=20000, population_size=200, max_knockouts=5,
                     plot=False, growth_coupled=True, seed=42)
result

Plot the production envelope of the first design (index 0) together with the wild type. The mutant's envelope should have a lower bound above zero at its maximum growth rate.

In [ ]:
result.plot(plotter, 0)

Show the simulated flux distribution of the mutant on the Escher map of central metabolism.

In [ ]:
result.display_on_map(0, "e_coli_core.Core metabolism")

## OptKnock

OptKnock uses a bilevel mixed-integer linear programming approach to identify reaction knockouts ([Burgard et al., 2003](https://doi.org/10.1002/bit.10803)):

$$
\begin{matrix}
maximize & \mathit{v_{chemical}} & & (\mathbf{OptKnock}) \\
\mathit{y_j} & & & \\
subject~to & maximize & \mathit{v_{biomass}} & (\mathbf{Primal}) \\
& \mathit{v_j} & & & & \\
\end{matrix}\\
\begin{bmatrix}
subject~to  & \sum_{j=1}^{M}S_{ij}v_{j} = 0,\\ 
& v_{carbon\_uptake} = v_{carbon~target}\\ 
& v_{apt} \ge v_{apt\_main}\\ 
& v_{biomass} \ge v_{target\_biomass}\\ 
& v_{j}^{min} \cdot y_j \le v_j \le v_{j}^{max} \cdot y_j, \forall j \in \boldsymbol{M} \\
\end{bmatrix}\\
\begin{align}
 & y_j = {0, 1}, & & \forall j \in \boldsymbol{M} & \\
 & \sum_{j \in M} (1 - y_j) \le K& & & \\
\end{align}
$$



Here $y_j$ is a binary variable per reaction (1 = present, 0 = knocked out), $S$ is the stoichiometric matrix and $K$ is the maximum number of knockouts. The outer problem chooses $y$ to maximise the chemical's flux, while the inner (primal) problem lets the cell maximise its growth. `fraction_of_optimum=0.1` requires the mutant to keep at least 10 % of the wild-type growth rate.

In [ ]:
from cameo.strain_design.deterministic.linear_programming import OptKnock

In [ ]:
optknock = OptKnock(model, fraction_of_optimum=0.1, exclude_reactions=['ATPS4r', 'CO2t', 'H2Ot'])

Again, we put a few reactions on the exclude list to avoid uninteresting solutions (ATP synthase, and CO₂ and water transport). The run time grows quickly with `max_knockouts`: with a commercial MILP solver three knockouts take seconds to minutes, with GLPK expect a few minutes. More knockouts can take hours.

Each row is an optimal knockout set with the predicted product and biomass fluxes and the FVA range of the target at that biomass.

In [ ]:
result = optknock.run(max_knockouts=3, target="EX_ac_e", biomass="BIOMASS_Ecoli_core_w_GAM")

In [ ]:
result

In [ ]:
result.plot(plotter, 0)

In [ ]:
result.display_on_map(0, "e_coli_core.Core metabolism")

## Exercise (20 min)

* Use OptGene or OptKnock to find a growth-coupled design for a product of your choice, for example D-lactate (`EX_lac__D_e`) under anaerobic conditions.

In [ ]:
# Your code here


## References

- Burgard, Pharkya, Maranas (2003). Optknock: A bilevel programming framework for identifying gene knockout strategies for microbial strain optimization. *Biotechnology and Bioengineering* 84:647–657. [doi.org/10.1002/bit.10803](https://doi.org/10.1002/bit.10803)
- Cardoso, Jensen, Lieven et al. (2018). Cameo: A Python Library for Computer Aided Metabolic Engineering and Optimization of Cell Factories. *ACS Synthetic Biology* 7:1163–1166. [doi.org/10.1021/acssynbio.7b00423](https://doi.org/10.1021/acssynbio.7b00423)
- Patil, Rocha, Förster et al. (2005). Evolutionary programming as a platform for in silico metabolic engineering. *BMC Bioinformatics* 6:308. [doi.org/10.1186/1471-2105-6-308](https://doi.org/10.1186/1471-2105-6-308)